In [2]:
# =============================================================================
# OPERAÇÃO 0.81+: RANDOM FOREST COM RUL CLIPPING E PH ESTABILIZADO
# =============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

print("📥 Carregando o dataset limpo (sem redundâncias)...")

# --- PASSO 0: CARREGAMENTO DOS DADOS LIMPOS ---
# Certifique-se que o ficheiro 'dataset_limpo.csv' está na mesma pasta
try:
    df_master = pd.read_csv('dataset_limpo.csv')
except FileNotFoundError:
    print("❌ ERRO: O ficheiro 'dataset_limpo.csv' não foi encontrado!")
    print("Verifique se correu o notebook de limpeza até ao fim e se o ficheiro está na mesma pasta.")
    raise

# --- PASSO 1: RECONSTRUÇÃO DAS MATRIZES ---
print("⚙️ Preparando as matrizes de treino e validação...")

# Separar os grupos
df_tr = df_master[df_master['Grupo'] == 'Treino']
df_vl = df_master[df_master['Grupo'] == 'Validação']

# Identificar as features (tudo o que não é ID, Grupo ou Alvo)
features = [col for col in df_master.columns if col not in ['ID_Trem', 'Ciclo', 'Grupo', 'RUL_Alvo']]

# Criar os Xs (Variáveis de entrada)
X_train_slim = df_tr[features].fillna(0)
X_val_slim = df_vl[features].fillna(0)

# Criar os Ys (Alvos Originais 0-400)
y_train = np.clip(df_tr['RUL_Alvo'].values, 0, 400)
y_val = np.clip(df_vl['RUL_Alvo'].values, 0, 400)


# --- PASSO 2: RUL CLIPPING (O Segredo da Performance) ---
print("🎯 Aplicando RUL Clipping (Foco na Degradação Final)...")
limite_rul = 125
y_train_clipped = np.clip(y_train, 0, limite_rul)
y_val_clipped = np.clip(y_val, 0, limite_rul)


# --- PASSO 3: TREINO DO RANDOM FOREST DE ELITE ---
modelo_elite = RandomForestRegressor(
    n_estimators=500,        # Mais árvores para estabilizar a previsão
    max_depth=12,            # Altura controlada para generalizar melhor
    min_samples_leaf=3,      # Evita que o modelo decore ruídos
    max_features='sqrt',     # Reduz a dependência de uma só variável
    random_state=42,
    n_jobs=-1
)

print("🚂 Treinando com Foco no Horizonte de Falha...")
modelo_elite.fit(X_train_slim, y_train_clipped)


# --- PASSO 4: PREVISÃO E ESTABILIZAÇÃO DE ELITE (FOCO EM PH) ---
# Usamos o df_vl para manter a estrutura dos comboios de validação
df_final_elite = pd.DataFrame({'ID_Trem': df_vl['ID_Trem'], 'Ciclo_Atual': df_vl['Ciclo'], 'RUL_Real': y_val_clipped})
df_final_elite['Prev_Raw'] = modelo_elite.predict(X_val_slim)

# Filtro mais forte (Span 10) para evitar que oscilações quebrem o PH
df_final_elite['RUL_Smooth'] = df_final_elite.groupby('ID_Trem')['Prev_Raw'].transform(
    lambda x: x.ewm(span=10, adjust=False).mean()
)

# Forçar a curva a ser estritamente decrescente (Física da Falha)
df_final_elite['RUL_Final'] = df_final_elite.groupby('ID_Trem')['RUL_Smooth'].cummin()


# --- PASSO 5: CÁLCULO DE MÉTRICAS OFICIAIS ---
def calcular_metricas_clipped(df, col):
    resultados = []
    for trem, dados in df.groupby('ID_Trem'):
        y_r, y_p = dados['RUL_Real'].values, dados[col].values
        rmse = np.sqrt(np.mean((y_r - y_p)**2))
        ss_res, ss_tot = np.sum((y_r - y_p)**2), np.sum((y_r - np.mean(y_r))**2)
        score = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        acertos = np.abs(y_r - y_p) <= 15
        acc = np.mean(acertos) * 100
        ph = 0
        for a in reversed(acertos):
            if a: ph += 1
            else: break
        resultados.append({'Score': score, 'RMSE': rmse, 'Acc': acc, 'PH': ph})
    return pd.DataFrame(resultados)

res = calcular_metricas_clipped(df_final_elite, 'RUL_Final')


# --- PASSO 6: TABELA CONSOLIDADA DE PERFORMANCE ---
score_medio = res['Score'].mean()
rmse_medio = res['RMSE'].mean()
acc_media = res['Acc'].mean()
ph_medio = res['PH'].mean()

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Métrica</b>", "<b>Resultado Médio (Validação)</b>"],
        fill_color='#001F3F',
        font=dict(color='white', size=14),
        align='center'
    ),
    cells=dict(
        values=[
            ["SCORE (R²)", "RMSE", "ACCURACY (%)", "PH (HORIZONTE)"],
            [f"{score_medio:.4f}", f"{rmse_medio:.2f}", f"{acc_media:.2f}%", f"{ph_medio:.1f} ciclos"]
        ],
        fill_color=[['#f2f6f9', '#ffffff']*2],
        font=dict(color='black', size=13),
        align='center',
        height=35
    )
)])

fig_table.update_layout(
    title_text=f"<b>🏆 Veredito Final: Random Forest com Dados Limpos e Clipping ({limite_rul} ciclos)</b>",
    title_x=0.5,
    margin=dict(l=10, r=10, t=50, b=10)
)

print(f"\n✅ Relatório gerado! Score Final: {score_medio:.4f}")
print(f"✅ PH Médio atualizado: {ph_medio:.1f} ciclos")
fig_table.show(renderer='browser')

📥 Carregando o dataset limpo (sem redundâncias)...
⚙️ Preparando as matrizes de treino e validação...
🎯 Aplicando RUL Clipping (Foco na Degradação Final)...
🚂 Treinando com Foco no Horizonte de Falha...

✅ Relatório gerado! Score Final: 0.8275
✅ PH Médio atualizado: 0.0 ciclos
